# Exploratory Data Analysis & Preprocessing

In this notebook, we will demonstrate how to go from raw, uncleaned data to preparing clean features for machine learning. We will use the Pima Indian Diabetes dataset to demonstrate automated profiling, data cleaning, outlier handling, and standardization.

### 1. Data Loading and Initial Inspection

First, we import the necessary libraries (such as Pandas, NumPy, Seaborn, and Matplotlib) and load our raw dataset (`diabetes.csv`). We display the first few rows to understand the structure of the data.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Locate and load the dataset
path = os.path.join('..', '..', 'Datasets', 'diabetes.csv')
if not os.path.exists(path):
    path = os.path.join('Datasets', 'diabetes.csv')
df = pd.read_csv(path)
df.head()

### 2. Automated Data Profiling

We use `ydata-profiling` to generate a comprehensive automated HTML report. This profile report will help us analyze the data types, distributions, correlations, and highlight key issues (like missing or zero values) in our dataset.

In [ ]:
from ydata_profiling import ProfileReport

report = ProfileReport(df, title="Diabetes Dataset Profile Report")
report.to_file("profile_report.html")

### 3. Data Cleaning

Based on our profiling, we perform the following cleaning steps:
1. **Invalid Zeros**: In the Pima Indian Diabetes dataset, columns such as `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` contain zero values. A value of `0` is physiologically impossible for these features. We replace them with `NaN` to indicate missing data.
2. **Imputation**: We fill the missing values (`NaN`) with the median of their respective columns to avoid bias.
3. **Duplicates**: We check for and remove any duplicate rows to ensure data integrity.

In [ ]:
# Replace invalid 0 values with NaN in the selected columns
cols_with_zero_invalid = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_zero_invalid] = df[cols_with_zero_invalid].replace(0, np.nan)

# Display the count of missing values per column before imputation
print("Missing values count before imputation:")
print(df.isnull().sum())

# Impute missing values with column-wise medians
for col in cols_with_zero_invalid:
    df[col] = df[col].fillna(df[col].median())

# Check and remove duplicate rows
duplicate_count = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicate_count}")
if duplicate_count > 0:
    df.drop_duplicates(inplace=True)

# Verify cleaning results
print("\nMissing values count after cleaning:")
print(df.isnull().sum())
df.head()

### 4. Data Scaling and Preprocessing (Standardization)

Since the numerical features in our dataset have varying ranges (e.g., `Insulin` values are much larger than `DiabetesPedigreeFunction` values), we scale them. Standardizing the features using `StandardScaler` sets their mean to 0 and variance to 1, ensuring all features contribute equally to models and visualizations.

In [ ]:
# Separate features and target
X = df.drop(columns=['Outcome'])
y = df['Outcome']

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for clean visualization
df_scaled = pd.DataFrame(X_scaled, columns=X.columns)
df_scaled['Outcome'] = y.values
df_scaled.head()